In [ ]:
%matplotlib inline

In [ ]:
#Import Libraries here

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio
import os
import sys
pio.renderers.default = "notebook_connected"
%matplotlib inline

In [ ]:
current_dir = os.path.abspath('')
project_root = os.path.abspath(os.path.join(current_dir, '../../'))

if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)

print(f"Working directory set to: {os.getcwd()}")

In [ ]:
#Import project modules here

import importlib
from src.portfolio_simulation_utils import portfolio_simulation as p_sim
from src.data_pipeline_utils import data_fetching_handling as data_pipe
import src.plotting_utils.plotting_utils as plot_utils
importlib.reload(p_sim)
importlib.reload(data_pipe)

# Project Overview

This project explores how the core mathematical concepts we explored in our course, including almost all of the topics we studied in the lectures for linear algebra, calculus, probabilities, and statistics, are applied in quantitative finance.

The presentation in this project focuses on three financial topics:

- Portfolio optimization (Markowitz Modern Portfolio Theory)
- Option pricing (Black–Scholes model)
- Fixed income instruments with embedded options

I would like to demonstrate how the same mathematical tools appear repeatedly across these topics: matrix algebra, probability distributions, derivatives of functions, and statistics. The project, therefore, attempts to study how these tools allow us to model financial risk and portfolio behavior and how mathematics is at the core of this aspect in the profession of a finance expert.

# Central Question

How are mathematical tools such as matrix and vector algebra, probability theory, statistics, and calculus be used to model risk, price options, and optimize financial portfolios? What value do they add?

# Thesis

Financial risk is fundamentally nonlinear. While linear approximations such as expected return or bond durations provide an idea at a certain point of these nonlinear functions, the true behavior of financial systems is driven by second-order structures such as variance, covariance, convexity, and option gamma.

This project will attempt to demonstrate how these nonlinear effects emerge in three different settings and demonstrate how solid finance theory and mathematics merge together to find practical applications in:

1. Portfolio optimization
2. Option pricing
3. Fixed income instruments with embedded options

These three topics are by far non-exhaustive examples of this phenomenon, but I will attempt to make an engaging and informative showcase in this process, where:

1. I will demonstrate how all the concepts without exception in our course have a direct and practical financial application in today's environemnt, and
2. How mathematical concepts fundamentally transormed and shaped the understading of finance, financial instruments and capital markets in the past 75 years in order to become the pillars of contemporary investment practice

# Methodology

The project follows three analytical approaches:

1. Empirical simulation  
   Monte Carlo simulation is used to explore the feasible set of portfolio risk–return combinations.

2. Analytical derivation  
   Closed-form mathematical models such as the Markowitz efficient frontier and the Black–Scholes option pricing formula are examined.

3. Numerical optimization  
   Constrained optimization methods are used to compute optimal portfolios subject to realistic investment constraints.

The main mathematical tools used include:

- Linear algebra (matrix notation and covariance matrices)
- Probability theory (normal distributions and cumulative distribution functions)
- Calculus (first and second derivatives)
- Taylor approximations

# Project Structure

The project is organized into five notebooks:

**1_1 – Efficient Frontier (Practical Approach)**  
Monte Carlo simulation of random portfolios to visualize the feasible region.

**1_2 – Efficient Frontier (Mathematical Approach)**  
Derivation of the efficient frontier using the Markowitz closed-form solution.

**1_3 – Efficient Frontier (Algorithmic Approach)**  
Numerical optimization of portfolio weights under constraints.

**2_1 – Black–Scholes and Portfolio Protection**  
Application of option pricing to hedge portfolio downside risk.

**2_2 – Option Pricing in Bonds**  
Application of option theory to callable bonds and convexity effects.

Notebooks 1_1 to 1_3 demonstrate modern portfolio theory and show three different approaches to finding efficient portfolios. I have designed the notebooks so that it would make the best sense to go through them in the exact sequence. Notebooks 2_1 and 2_2 pivot to option pricing, where I will attempt to demonstrate that while seemingly different, the math concepts we studies in linear algebra and calculus directly translate into option pricing as well - I study options in two separate areas - equities and fixed income instruments. 

So, let's start with the first topic. 

### 1. Let us first choose our stocks and extract the necessary data utilizing the yfinance library in Python ###

In the list ticker below, we can include the symbol tickers of stocks we like. Since this is more of a scientific experiment and not a data science project, **let us stick to stocks from the US equity market only** in order not to deal with mismatching working days. I have selected ten of the stocks I monitor daily for different reasons as a default structure.

For each ticker, we start by calculating the daily returns  with the utility methods in the data pipeline module. Afterwards, we have to make sure that the result of the print is 1. This means that each ticker has a price history for the full selected period, which by default is going to be 10 years.

If the result is larger than 1, then one or more tickers do not have a full period of trading data, and you must inspect the values in the all_df_shapes dictionary and replace the tickers with shorter time periods or rerun the same list of tickers by selecting a shorter period of time, which is less preferable for this theoretical experiment. 

In [ ]:
tickers = ['AAPL', 'NVDA', 'MSFT', 'JNJ', 'BAC', 'VZ', 'WMT', 'UPS', 'PFE', 'JPM']
all_df_shapes = {}

for ticker in tickers:
    data = data_pipe.save_10_year_single_stock_data_to_csv(ticker)
    return_data = data_pipe.create_returns_and_save(data, ticker)
    all_df_shapes[ticker] = return_data.shape

all_same = set(all_df_shapes.values())
print(f"{len(all_same)} - a result of 1 is a green light to go as all datasets are complete, if not go back and choose different stocks.")

### Let us randomly check if we have what we need ###

I will pull the different types of data that was created in the above steps. This is a necessary sanity check to investigate manually what we achieved. You can go ahead and see the actual CSV files that now fill the newly created data folder in the main project folder.

In [ ]:
MSFT_data = data_pipe.fetch_raw_data('MSFT')
return_data_PFE = data_pipe.fetch_returns_data('PFE')

print(return_data_PFE)

In [ ]:
fig = plot_utils.create_candlestick_graph('MSFT')
fig.show()

In [ ]:
MSFT_return_data = data_pipe.fetch_returns_data('MSFT')
mean = MSFT_return_data["log_return"].mean()
variance = MSFT_return_data["log_return"].var()
st_dev = MSFT_return_data["log_return"].std()

print(f"Mean log return in observation period: {mean},\n"
      f"Variance of log returns in observation period: {variance},\n"
      f"Standard deviation of log returns in observation period: {st_dev}")

ticker = 'MSFT'
fig = plot_utils.create_histogram_distribution_daily_log_returns(MSFT_return_data, ticker, mean, st_dev)
plt.show()

In [ ]:
pd.DataFrame(MSFT_return_data["log_return"]).describe().T

Skewness measures the asymmetry of the return distribution, while kurtosis measures the thickness of the distribution tails relative to a normal distribution. We can check our selected ticker's distribution characteristics. Equity returns over long periods typically are with slightly negative skewness, which means fatter (more probability) left tail and positive kurtosis, which means fatter tails in general (more probability in the tails). Let's see our current result, the exercise can be repeated for each stock we selected above my manipulating the code above.  

In [ ]:
skewness = MSFT_return_data["log_return"].skew()
kurtosis = MSFT_return_data["log_return"].kurt()

print(f"Distribution of {ticker} returns - Skewness: {skewness:.4f}, Kurtosis: {kurtosis:.4f}")

### Covariance Matrix of Stock Returns ###
The matrix $X$ represents the observed returns of two assets over $\text{n}$ time periods.  
Each row corresponds to a specific observation in time, while each column corresponds to a different asset.  
$$ X =
\begin{bmatrix}
r_{1,1} & r_{2,1} \\
r_{1,2} & r_{2,2} \\
\vdots  & \vdots  \\
r_{1,n} & r_{2,n}
\end{bmatrix} $$

The element $r_{i,t}$ denotes the return of asset $\text{i}$ in period $\text{t}$.

From this matrix of returns we compute the **covariance matrix** $ \Sigma $, which measures how the returns of the assets move together. In portfolio theory, covariance captures the degree to which two assets tend to increase or decrease simultaneously.

The covariance matrix is computed using the centered return matrix $X - \bar{X}$, where $ \bar{X} $ is the matrix of mean returns for each asset. Subtracting the mean removes the average level of returns and isolates the fluctuations around that mean.

The expression:
$$ \Sigma = \frac{1}{n-1}(X - \bar{X})^\top (X - \bar{X}) $$

computes the sample covariance matrix of the asset returns. The transpose operator $ (\cdot)^\top $ converts the return matrix so that the multiplication produces all pairwise covariances between the assets. The normalization factor $ \frac{1}{n-1} $ ensures an unbiased estimator of the covariance when working with sample data.

In the context of portfolio optimization, the covariance matrix is a fundamental object because it determines the **risk of any portfolio combination** through the quadratic form $w^\top \Sigma w $, where $\text{w}$ is the vector of portfolio weights.

In [ ]:
returns_df =  data_pipe.build_returns_df(tickers)

print(returns_df)

cov_matrix = returns_df.cov()
print(cov_matrix)

eigenvalues = np.linalg.eigvals(cov_matrix)
print(eigenvalues)

### Correlation (Normalized Covariance) ###
$$ \rho_{ij} = \frac{\mathrm{Cov}_{ij}}{\sigma_i \sigma_j} $$

Correlation is the **normalized form of covariance**, measuring the strength of the linear relationship between two asset returns independently of their scale. By dividing the covariance $ \mathrm{Cov}_{ij} $ by the product of the individual volatilities $ \sigma_i \sigma_j $, the result becomes dimensionless and bounded between $-1$ and $1$.

In [ ]:
corr_matrix = returns_df.corr()
print(corr_matrix)

### Let us plot the correlation matrix ###

In the portfolio theory, this graph is very is very informative. It points our attention visually to couples of stocks with stock prices, which are highly correlated - in other words their prices move very often in the same direction and on the same scale. When stocks are highly correlated, the benefits of diversification are lower and vice versa. 

In [ ]:
fig = plot_utils.create_correlation_heatmap(corr_matrix)
plt.show()

### Set up an optimal portfolio simulation calculation ###

In [ ]:
annual_returns = returns_df.mean() * 252
annual_cov_matrix = returns_df.cov() * 252
risk_free_rate = 0.03
sim_runs = 100000 #This is a crucial constant. See explanation below. At default, I put 100 000 runs
n = len(returns_df.columns)

# I will set up placeholders to store all outputs of the simulation calculation:
weights_runs = np.zeros((sim_runs, n))
sharpe_ratio_runs = np.zeros(sim_runs)
expected_portfolio_returns_runs = np.zeros(sim_runs)
volatility_runs = np.zeros(sim_runs)

## Run the calculations to obtain an efficient frontier and the optimal portfolio

The Monte Carlo simulation approach used to generate the efficient frontier in this notebook was inspired by the *CFA Institute Python Programming Fundamentals* Practical Skills Module taught by Dr. Ryan Ahmed.

I have also implemented a similar Monte Carlo efficient frontier workflow in a separate project: my Django Advanced final project, **Equity Optimizer App** (GitHub: https://github.com/Kamend1/equity-optimizer-app).

A key benefit of the *Math for Developers* course is that it improved my understanding of vectors, matrices, and quadratic forms. After revisiting my earlier implementation, I realized that the simulation engine performed redundant matrix/vector operations. Refactoring the code to remove those unnecessary operations significantly improved performance. In the earlier version, approximately 30,000 simulations required ~20–30 minutes. After the refactor, the engine can run 1,000,000 simulations in under ~20 minutes on the same machine. This matters because a denser Monte Carlo cloud produces a smoother and more informative approximation of the feasible set and the efficient frontier.

In the setup section above, the parameter `sim_runs` controls the number of random portfolios generated. For development and quick validation, `100_000` runs are sufficient. This should take around 2 minutes on most regular computers. For final charts and conclusions, `1_000_000` runs produce cleaner frontier approximation and we can see later in notebook 1_3 that the more the simulation runs, the better our guess at the characteristics of an efficient portfolio. This is called the **law of large numbers**. If you have computing power, ram and time, try even larger number of guesses or sim_runs.

In [ ]:
for i in range(sim_runs):
    # Generate random weights
    weights = p_sim.generate_portfolio_weights(n)
    
    # Store the weights
    weights_runs[i, :] = weights

    # Call "simulation_engine" function and store Sharpe ratio, return, and volatility
    expected_portfolio_returns_runs[i], volatility_runs[i], sharpe_ratio_runs[i], \
     = p_sim.simulation_engine(weights, annual_returns, annual_cov_matrix, risk_free_rate)

    #Let's monitor progress by doing prints on every 250th run
    if i % 250 == 0:
        print(f"Simulation Run = {i}")
        print(f"Weights = {weights_runs[i].round(3)},\n Sharpe Ratio = {sharpe_ratio_runs[i]:.5f},\n"
             f"Expected return = {expected_portfolio_returns_runs[i]},\n"
             f"Volatility = {volatility_runs[i]}")
        print('\n')

### In this step, we extract the output data produced in the simulation calculation ###

- There is a problem - what if Sharpe Ratio is a negative?! #TODO

In [ ]:
sim_out_df = pd.DataFrame({'Volatility': volatility_runs.tolist(), 
                           'Portfolio_Return': expected_portfolio_returns_runs.tolist(), 
                           'Sharpe_Ratio': sharpe_ratio_runs.tolist() })
print(sim_out_df)

max_sharpe_idx = sim_out_df["Sharpe_Ratio"].idxmax()
optimal_portfolio = sim_out_df.loc[max_sharpe_idx]
cloud_df = sim_out_df.drop(index=max_sharpe_idx)
optimal_portfolio_weights = weights_runs[max_sharpe_idx]

print(optimal_portfolio)

### We will now plot the results and make some interesting observations ###

The scatter plot will generally have a parabolic shape. The upper boundary should be colored in bright yellow and it will approximate the shape of the frontier of portfolios with efficient returns for a unit of risk. Our guess for the optimal portfolio will lie on that frontier - a bright red dot. 

Check out the volatility and return results for portfolios generated in the sim runs.

In [ ]:
fig = plot_utils.create_sim_output_scatter(cloud_df, optimal_portfolio)
fig.show()

fig_1, fig_2, fig_3 = plot_utils.sim_results_plot(sim_out_df)

fig_1.show()
fig_2.show()
fig_3.show()

### Finally, let's print and take not of the optimal portfolio weights ###

We are going to use the outcome from this exercise for comparison reasons in notebook 1_3 down the road.

In [ ]:
print(list(zip(tickers, optimal_portfolio_weights)))

### Conclusion ###

So far we have used very little math. Most of us in the finance profession are taught to understand intuitively the concept and find practical solutions, without fully understanding the linear algebra and the geometry behind it. I substituted the actual solution produced by Markowitz in 1952, where he optimized to find the portfolio weights of a portfolio including predefined risk-bearing assets by finding the lowest-variance portfolio for each targeted level for portfolio return.

Instead of solving, I just produced $\text{n}$ different portfolios by randomly generating weights, calculted their variance and returns and then plotted each return-variance combination on a scatter plot. The bright yellow dots appear to form a parabolic shape, which actually defines the efficient frontier defined by Markowitz. The greater n is in term of simulation runs, the closer the result is to the actual outcome. I will leave the variable sim_runs at 100 000, but you can try even higher. For example I tried 1 000 000 runs, it took around 20 minutes to calculate. But this is brute force, not mathematics. 

Let's now move to notebook 1_2 in order to observe how the theory and its supporting mathematics actually work. 